# Parsing

Normalizing model output and getting structured data out of it. Pure: standard library and pydantic
only, no langchain, no config. Callers: `%run ./parsing`.

In [ ]:
from pydantic import ValidationError

# reasoning models return their thinking as content blocks; drop them
_SKIP_BLOCKS = {"reasoning", "thinking", "redacted_reasoning", "reasoning_content"}


def message_text(content):
    """Any model output -> str. Never returns None, never raises.

    The router picks a model per request, so the same call site sees a plain string from one model
    and a list of content blocks from the next. Call this on every model output.
    """
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        if content.get("type") in _SKIP_BLOCKS:
            return ""
        for key in ("text", "content"):
            if key in content:
                return message_text(content[key])
        return ""
    if isinstance(content, (list, tuple)):
        return "".join(message_text(block) for block in content)
    if hasattr(content, "content"):
        return message_text(content.content)
    return str(content)


# Databricks structured outputs take a subset of JSON Schema. Pydantic emits every one of these by
# default: Optional/Union -> anyOf, a nested model -> $ref/$defs, dict[str, str] ->
# additionalProperties. Keep schemas flat, every field required, meaning in the descriptions.
_UNSUPPORTED = ("anyOf", "oneOf", "allOf", "$ref", "$defs", "prefixItems", "pattern")
MAX_SCHEMA_KEYS = 64


def _walk(node):
    """Every (key, value) pair in a JSON Schema, at any depth."""
    if isinstance(node, dict):
        for key, value in node.items():
            yield key, value
            yield from _walk(value)
    elif isinstance(node, list):
        for value in node:
            yield from _walk(value)


def check_schema_supported(schema):
    """Returns the reasons `schema` would be rejected; empty list means it is usable."""
    spec = schema.model_json_schema()
    pairs = list(_walk(spec))
    keys = [key for key, _ in pairs]
    problems = [key for key in _UNSUPPORTED if key in keys]

    # additionalProperties needs its value, not just its name: `false` is what strict mode wants
    # and pydantic emits it for extra="forbid", while a dict field emits a schema there instead.
    # Rejecting the key outright would reject the correct form along with the wrong one.
    if any(key == "additionalProperties" and value is not False for key, value in pairs):
        problems.append("additionalProperties as a schema (a dict or a free-form object)")

    optional = sorted(set(spec.get("properties", {})) - set(spec.get("required", [])))
    if optional:
        problems.append(f"not required in strict mode: {optional}")
    if len(keys) > MAX_SCHEMA_KEYS:
        problems.append(f"{len(keys)} keys > {MAX_SCHEMA_KEYS}")
    return problems


def ask_structured(chat, system_prompt, user_content, schema, default):
    """One model call constrained to `schema`, validated by pydantic. Returns `default` on failure.

    The schema goes to the server as response_format, so the output is constrained rather than
    coaxed - no fence stripping, no hunting for a brace in prose. `default` is a risk decision and
    the most important argument at every call site.
    """
    bound = chat.bind(response_format={
        "type": "json_schema",
        "json_schema": {"name": schema.__name__, "schema": schema.model_json_schema(),
                        "strict": True},
    })
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content}]
    text = ""
    for _ in range(2):  # first attempt, then one correction carrying pydantic's own error
        try:
            text = message_text(bound.invoke(messages))
            return schema.model_validate_json(text)
        except ValidationError as err:
            faults = [(".".join(str(p) for p in e["loc"]), e["msg"]) for e in err.errors()]
            messages += [
                {"role": "assistant", "content": text[:500]},
                {"role": "user", "content": f"That did not validate: {faults}. "
                                            "Answer again with an object that satisfies the schema."},
            ]
        except Exception:  # network, timeout, refusal - a second call will not help
            break
    return default